# Tokenization & Vocabulary — Code Companion

This notebook accompanies **Topic: Tokenization & Vocabulary**.

We'll do two things:
1. **Build a Byte-Pair Encoding (BPE) tokenizer completely from scratch**, using only
   pure Python, so you can see exactly how the merge algorithm works, step by step, on a
   corpus small enough to read by eye.
2. **Use real tokenizers** (including Llama 3.1's) to see how different production
   tokenizers split the same sentence differently -- directly relevant to every other
   notebook in this course, since the tokenizer is loaded together with the model in
   every `FastLanguageModel.from_pretrained(...)` call.

No GPU needed for any of this.

## Part A — Byte-Pair Encoding (BPE), Built From Scratch

### 1. Start with characters

BPE begins by splitting every word in the training corpus into individual characters,
with a special end-of-word marker (`</w>`) so the algorithm can tell "est" at the end of
a word apart from "est" in the middle of one.

In [1]:
from collections import Counter

# A tiny toy corpus: word -> how many times it appears in our "training data"
corpus = {
    "low": 5,
    "lower": 2,
    "lowest": 6,
    "newer": 6,
    "wider": 3,
    "new": 2,
}

def word_to_symbols(word):
    return list(word) + ["</w>"]

vocab = {tuple(word_to_symbols(word)): freq for word, freq in corpus.items()}

print("Starting vocabulary (word -> symbol sequence : frequency):")
for symbols, freq in vocab.items():
    print(f"  {' '.join(symbols):25s} : {freq}")

Starting vocabulary (word -> symbol sequence : frequency):
  l o w </w>                : 5
  l o w e r </w>            : 2
  l o w e s t </w>          : 6
  n e w e r </w>            : 6
  w i d e r </w>            : 3
  n e w </w>                : 2


### 2. Count pair frequencies, then merge the most frequent pair — repeat

At each step, BPE counts every adjacent pair of symbols across the whole corpus, merges
the single most frequent pair into a new symbol, and repeats. The ordered list of merges
*is* the trained tokenizer.

In [2]:
def get_pair_counts(vocab):
    pairs = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_pair(pair, vocab):
    new_vocab = {}
    bigram, replacement = " ".join(pair), "".join(pair)
    for symbols, freq in vocab.items():
        symbol_str = " ".join(symbols).replace(bigram, replacement)
        new_vocab[tuple(symbol_str.split(" "))] = freq
    return new_vocab

def train_bpe(vocab, num_merges):
    vocab = dict(vocab)
    merges = []
    for step in range(num_merges):
        pairs = get_pair_counts(vocab)
        if not pairs:
            break
        best_pair = pairs.most_common(1)[0][0]
        vocab = merge_pair(best_pair, vocab)
        merges.append(best_pair)
        print(f"Merge {step + 1}: {best_pair} -> '{''.join(best_pair)}'  "
              f"(occurred {pairs[best_pair]} times)")
    return vocab, merges

final_vocab, merges = train_bpe(vocab, num_merges=10)

print("\nFinal tokenized vocabulary:")
for symbols, freq in final_vocab.items():
    print(f"  {' '.join(symbols):25s} : {freq}")

Merge 1: ('w', 'e') -> 'we'  (occurred 14 times)
Merge 2: ('l', 'o') -> 'lo'  (occurred 13 times)
Merge 3: ('r', '</w>') -> 'r</w>'  (occurred 11 times)
Merge 4: ('lo', 'we') -> 'lowe'  (occurred 8 times)
Merge 5: ('n', 'e') -> 'ne'  (occurred 8 times)
Merge 6: ('w', '</w>') -> 'w</w>'  (occurred 7 times)
Merge 7: ('lowe', 's') -> 'lowes'  (occurred 6 times)
Merge 8: ('lowes', 't') -> 'lowest'  (occurred 6 times)
Merge 9: ('lowest', '</w>') -> 'lowest</w>'  (occurred 6 times)
Merge 10: ('ne', 'we') -> 'newe'  (occurred 6 times)

Final tokenized vocabulary:
  lo w</w>                  : 5
  lowe r</w>                : 2
  lowest</w>                : 6
  newe r</w>                : 6
  w i d e r</w>             : 3
  ne w</w>                  : 2


### 3. Applying learned merges to a brand-new word

Once we have the ordered list of merges, we can tokenize *any* new word — even one the
tokenizer has never seen — by applying the same merges in the same order.

In [3]:
def apply_bpe(word, merges):
    symbols = word_to_symbols(word)
    for pair in merges:
        i, new_symbols = 0, []
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

for test_word in ["lowest", "newest", "wider", "slow"]:
    print(f"{test_word:10s} -> {apply_bpe(test_word, merges)}")

lowest     -> ['lowest</w>']
newest     -> ['newe', 's', 't', '</w>']
wider      -> ['w', 'i', 'd', 'e', 'r</w>']
slow       -> ['s', 'lo', 'w</w>']


`"newest"` was **never** in our training corpus, yet BPE still splits it into sensible,
previously-seen pieces (like `new` + `est</w>`) instead of failing outright — this is the
"subword fallback" behavior that solves the out-of-vocabulary problem, and exactly why
every model we fine-tuned in this course can handle words it never saw during training.

## Part B — Real Tokenizers, Including Llama 3.1's

The from-scratch version above is a simplified teaching version. Production tokenizers
(Llama's BPE, BERT's WordPiece, T5's SentencePiece) use the same core idea but are
trained on billions of words.

> **Note:** the cells below need `pip install transformers` and internet access (the
> first call downloads each tokenizer's vocabulary file). Run this section in an
> environment with internet access, e.g. Google Colab.

In [ ]:
from transformers import AutoTokenizer

sentence = "Tokenization is fascinating, isn't it?"

tokenizer_names = {
    "Llama 3.1 (8B)": "unsloth/Meta-Llama-3.1-8B",   # the model used throughout this course
    "BERT (WordPiece)": "bert-base-uncased",
    "GPT-2 (byte-level BPE)": "gpt2",
    "T5 (SentencePiece)": "t5-small",
}

for label, checkpoint in tokenizer_names.items():
    tok = AutoTokenizer.from_pretrained(checkpoint)
    tokens = tok.tokenize(sentence)
    print(f"{label}")
    print(f"  tokens ({len(tokens)}): {tokens}")
    print()

This is the exact tokenizer that gets loaded alongside the model in every
`FastLanguageModel.from_pretrained("unsloth/Meta-Llama-3.1-8B")` call in the LoRA/QLoRA
and DPO notebooks — now you can see precisely what it does to raw text before any of that
training code runs.

### Token count vs. word count

A practical exercise matching the slides: token count is **not** the same as word count,
and that difference drives both API cost and context-window limits.

In [ ]:
paragraph = (
    "Internationalization and interoperability are notoriously long words "
    "that subword tokenizers love to split into several pieces."
)

word_count = len(paragraph.split())

for label, checkpoint in tokenizer_names.items():
    tok = AutoTokenizer.from_pretrained(checkpoint)
    token_count = len(tok.tokenize(paragraph))
    print(f"{label:22s} words: {word_count:3d}   tokens: {token_count:3d}   "
          f"ratio: {token_count / word_count:.2f} tokens/word")

### Tokenizing an Alpaca-formatted example

Let's tie this directly to the Data Preparation notebook: tokenize a *fully formatted*
Alpaca-style training example and confirm the EOS token is actually present as its own
token at the end.

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

llama_tokenizer = AutoTokenizer.from_pretrained("unsloth/Meta-Llama-3.1-8B")
EOS_TOKEN = llama_tokenizer.eos_token

formatted_example = alpaca_prompt.format(
    "Give three tips for staying healthy.", "",
    "1. Eat a balanced diet. 2. Exercise regularly. 3. Get enough sleep."
) + EOS_TOKEN

tokens = llama_tokenizer.tokenize(formatted_example)
print(f"Total tokens: {len(tokens)}")
print(f"Last 3 tokens: {tokens[-3:]}")
print(f"\nEOS token {EOS_TOKEN!r} present at the end: {EOS_TOKEN in tokens[-1] or tokens[-1] == EOS_TOKEN}")

## Recap & Try It Yourself

You just:
- Trained a Byte-Pair Encoding tokenizer completely from scratch and watched it merge
  characters into subwords step by step.
- Applied the learned merges to tokenize a brand-new word never seen during "training".
- Compared how Llama 3.1, BERT, GPT-2, and T5's real tokenizers split the same sentence.
- Measured the token-count-vs-word-count gap that drives both cost and context limits.
- Confirmed the EOS token is present as a real, distinct token in a fully formatted
  Alpaca training example.

**Things to try:**
1. Increase `num_merges` in Part A from 10 to 30 and see how the final vocabulary changes.
2. Add more words to the toy `corpus` and see how the merge order shifts.
3. In Part B, try a sentence in a language other than English, or with emojis, and
   compare token counts across all four tokenizers.
4. Look up `tokenizer.convert_tokens_to_ids(tokens)` on the Llama tokenizer to see the
   numeric IDs actually fed into the model during training.